# Bassik analysis

## Table of Contents:

* [0. Dependencies](#Dependencies)
* [1. Preparing the environment](#Preparing-the-environment)
* [2. Calculating predicted vs observed fold changes](#Calculating-predicted-vs-observed-fold-changes)
* [3. Running Bassik analysis](#Running-bassik-analysis)

***

## Dependencies

Please see [INSTALL_README.md](../INSTALL_README.md) for the installation of software dependencies. This Notebook assumes that [R](https://cran.r-project.org/) and library dependencies have been installed and that the `RScript` command is available with access to the relevant data.

***

## Preparing the environment

Several paths are used as input to more than one script. To simplify the commands and improve usability, commonly used paths are stored as environment variables. Those variables are then used in the script arguments to shorten input and output data paths. 

This Notebook assumes that you have the following directory structure and files in place before running the commands: 

* **Top level directory** (`REPO_PATH`)
    * **DATA**
        * **preprocessing**
            * *count_matrix.scaled.tsv*
            * *lfc_matrix.scaled.tsv*
        * **dual_guide/Bassik** (`BASSIK_PATH`)
        * **RDS/single_guide** (`RDS_PATH`)
    * **METADATA**
        * **libraries**
            * *dual_guide_matrix.tsv*
        * *sample_annotations.tsv*

The scripts require several files to be present:

* `METADATA/sample_annotations.tsv` - sample metadata.
* `METADATA/libraries/dual_guide_matrix.tsv` - mapping of each dual guide pair to its respective single guides

In [ ]:
# Set the top level path for the repository
export REPO_PATH=$(dirname `pwd`)

# Show the repository path
echo "Repository path: ${REPO_PATH}"

# Check the repository path exists (don't need to check subdirectories)
if [ ! -d "${REPO_PATH}" ]; then
  echo "Top level directory path does not exist: ${REPO_PATH}"
fi

Once the top level directory has been set (this will likely be the path to your clone of the repository), we then set several other resusable paths as environment variables and create their directories if they don't exist. It should not be assumed this exist when you clone the repository as they may be present in the .gitignore files (e.g. output logs or large data files).

In [ ]:
# Set environment variables for reusable paths
export BASSIK_PATH="${REPO_PATH}/DATA/dual_guide/Bassik"
export RDS_PATH="${REPO_PATH}/DATA/RDS/dual_guide"

# Show the paths (debug)
echo "Bassik output directory: ${BASSIK_PATH}"
echo "RDS output directory: ${RDS_PATH}"

# Create the directories if they don't exist 
mkdir -p "${BASSIK_PATH}"
mkdir -p "${RDS_PATH}"

***

## Calculating predicted vs observed fold changes

Before running the Bassik analysis, it is first necessary to calculate the predicted (sum of single guideA and guideB fold changes) and observed (dual guide fold change) for each guide pair. This is done using an LSF jobscript (`SCRIPTS/dual_guide/Bassik/01_get_obs_vs_pred_values_by_chunk.sh`) which calls an R script (`SCRIPTS/dual_guide/Bassik/01_get_obs_vs_pred_values_by_chunk.R`). The R script takes as input the dual guide matrix (`METADATA/libraries/dual_guide_matrix.tsv`) which was generated by `SCRIPTS/preprocessing/00_prepare_libraries.R` where each row represents a dual guide pair for given gene pair (`g1g2`) and it's constituent single guides (`g1` and `g2`). It then divides the matrix into chunks each of which contains a maximum of user-defined rows (`-n`) and selects the user-defined chunk (`-i`) for processing. 

To run the whole dual guide matrix at once (without needing to merge):

```
num_lines=$(wc -l ${REPO_PATH}/METADATA/libraries/dual_guide_matrix.tsv | cut -f1 -d ' ')
Rscript ${REPO_PATH}/SCRIPTS/dual_guide/Bassik/01_get_obs_vs_pred_values_by_chunk.R \
  --fc  ${REPO_PATH}/DATA/preprocessing/lfc_matrix.scaled.tsv \
  --annotations 13 \
  --doubles_guide_matrix ${REPO_PATH}/METADATA/libraries/dual_guide_matrix.tsv \
  --helper ${REPO_PATH}/SCRIPTS/dual_guide/helper.R \
  -o ${REPO_PATH}/DATA/dual_guide/Bassik \
  -n $num_lines \
  -i 1"
```

To run LSF jobscript:

In [ ]:
bsub < ${REPO_PATH}/SCRIPTS/dual_guide/Bassik/01_get_obs_vs_pred_values_by_chunk.R

Next, the chunks need to be merged into a single data frame for the Bassik analysis (`${REPO_PATH}/DATA/dual_guide/Bassik/pred_vs_obs_y12.tsv`) and a list of guide pairs with missing data due to previous filtering `${REPO_PATH}/DATA/dual_guide/Bassik/missing_data.tsv`.

In [ ]:
# Get a list of chunk files
find ${REPO_PATH}/DATA/dual_guide/Bassik -name "pred_vs_obs_y12.[0-9]*" > "${REPO_PATH}/LOGS/dual_guide/Bassik/list_of_pred_vs_obs_y12_files.txt"

# Merge chunks
#Rscript ${REPO_PATH}/SCRIPTS/dual_guide/Bassik/02_merge_pred_vs_obs_chunks.R \
#    -f ${REPO_PATH}/LOGS/dual_guide/Bassik/list_of_pred_vs_obs_y12_files.txt \
#    -p pred_vs_obs_y12 \
#    -o ${REPO_PATH}/DATA/dual_guide/Bassik \
#    -r ${REPO_PATH}/DATA/RDS/dual_guide/Bassik

# Get a list of chunk files
find ${REPO_PATH}/DATA/dual_guide/Bassik -name "missing_data.[0-9]*" > "${REPO_PATH}/LOGS/dual_guide/Bassik/list_of_missing_data_files.txt"

# Merge chunks
#Rscript ${REPO_PATH}/SCRIPTS/dual_guide/Bassik/02_merge_pred_vs_obs_chunks.R \
#    -f ${REPO_PATH}/LOGS/dual_guide/Bassik/list_of_missing_data_files.txt \
#    -p missing_data \
#    -o ${REPO_PATH}/DATA/dual_guide/Bassik \
#    -r ${REPO_PATH}/DATA/RDS/dual_guide/Bassik

*** 

## Running Bassik analysis

```
Rscript ${REPO_PATH}/SCRIPTS/dual_guide/Bassik/03_run_bassik_analysis.R \
    -f ${REPO_PATH}/DATA/preprocessing/lfc_matrix.scaled.tsv \
    -y ${REPO_PATH}/DATA/dual_guide/Bassik/pred_vs_obs_y12.tsv \
    -m ${REPO_PATH}/DATA/dual_guide/Bassik/missing_data.tsv \
    -s ${REPO_PATH}/METADATA/sample_annotations.tsv \
    --annotations 13 \
    -c "F1,F2,F3,F4,F5,F6,F7,F8,F9,F10" \
    -t ${REPO_PATH}/DATA/dual_guide/Bassik \
    -r ${REPO_PATH}/DATA/RDS/dual_guide/Bassik
```

In [8]:
bash ${REPO_PATH}/SCRIPTS/dual_guide/Bassik/03_run_bassik_analysis.sh

Repository path: /lustre/scratch124/casm/team113/users/vo1/5429_paralog_sl_vicky_finalised_for_paper
DONE.
